# 06 — 错误分析

**负责人**: 石韫嘉 | **周次**: W15
**目标**: 高误差样本分析、按价格区间分层误差分析、模型诊断

## 1. 环境与数据加载

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import compute_metrics, error_summary
from src.evaluation.visualization import price_bin_errors

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11})
sns.set_style('whitegrid')

RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

df_results = pd.read_csv(RESULTS_DIR / 'experiment_log.csv')
print('Environment ready')

In [ ]:
# Load data with consistent split
df = pd.read_csv(DATA_PROCESSED / 'florida_structured.csv')
target_col = 'lastSoldPrice'
drop_cols = [target_col, 'listPrice']
for col in ['_id', '_split', 'sanitized_text', 'clean_text', 'type', 'sub_type', 'zip', 'address', 'description']:
    if col in df.columns:
        drop_cols.append(col)
feature_cols = [c for c in df.columns if c not in drop_cols]

X_struct = np.nan_to_num(df[feature_cols].values.astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
y = df[target_col].values.astype(np.float64)
y = np.nan_to_num(y, nan=np.nanmedian(y))

tfidf = joblib.load(DATA_PROCESSED / 'florida_tfidf_features.pkl').astype(np.float64)
bert_emb = joblib.load(DATA_PROCESSED / 'florida_bert_embeddings.pkl').astype(np.float64)

indices = np.arange(len(y))
idx_train, idx_temp = train_test_split(indices, test_size=0.2, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42)

X_test = X_struct[idx_test]
y_test = y[idx_test]
X_text_test = {'tfidf': tfidf[idx_test].astype(np.float64), 'bert_embeddings': bert_emb[idx_test].astype(np.float64)}
print(f'Train={len(idx_train)}, Val={len(idx_val)}, Test={len(idx_test)}')

In [ ]:
# Load key models for error analysis
from src.models.structured_baseline import XGBoostBaseline, RandomForestBaseline
from src.models.text_baseline import TFIDFRidgeBaseline
from src.models.early_fusion import EarlyFusionMLP
from src.models.mid_fusion import MidFusionModel
from sklearn.preprocessing import StandardScaler

y_train_arr = y[idx_train]

# Mapping: display_name -> (class, modality, model_filename)
model_specs = [
    ('XGBoost', XGBoostBaseline, 'structured', 'XGBoostBaseline.joblib'),
    ('RandomForest', RandomForestBaseline, 'structured', 'RandomForestBaseline.joblib'),
    ('TFIDF+Ridge', TFIDFRidgeBaseline, 'text', 'TFIDFRidgeBaseline.joblib'),
    ('EarlyFusionMLP', EarlyFusionMLP, 'fusion_early', 'EarlyFusionMLP.joblib'),
    ('MidFusion', MidFusionModel, 'fusion_mid', 'MidFusionModel.joblib'),
]

predictions = {}
for name, cls, modality, filename in model_specs:
    model_path = MODELS_DIR / filename
    if not model_path.exists():
        print(f'Skip {name}: file {filename} not found')
        continue
    try:
        model = cls.load(str(model_path))
        # Repair _y_scaler for PyTorch models
        if hasattr(model, '_y_scaler') and model._y_scaler is None:
            model._y_scaler = StandardScaler()
            model._y_scaler.fit(y_train_arr.reshape(-1, 1))
        if modality == 'structured':
            pred = model.predict(X_test)
        elif modality == 'text':
            pred = model.predict(None, X_text=X_text_test)
        else:
            pred = model.predict(X_test, X_text=X_text_test)
        predictions[name] = pred
        m = compute_metrics(y_test, pred)
        print(f'{name}: RMSE=${m["rmse"]:,.0f}  R²={m["r2"]:.4f}  MAPE={m["mape"]:.1f}%')
    except Exception as e:
        print(f'{name}: error - {e}')

## 2. 按价格区间分层误差分析

In [ ]:
# ============================================================
# FIGURE 1: MAE and RMSE by price bin — all models
# ============================================================
N_BINS = 5
bin_results = {}
for name, pred in predictions.items():
    bin_results[name] = price_bin_errors(y_test, pred, n_bins=N_BINS)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_err = {'XGBoost': '#4472C4', 'RandomForest': '#5B9BD5', 'TFIDF+Ridge': '#ED7D31',
              'EarlyFusionMLP': '#70AD47', 'MidFusion': '#A855F7'}

for ax_idx, (metric, label) in enumerate([('mae', 'MAE ($)'), ('rmse', 'RMSE ($)')]):
    ax = axes[ax_idx]
    n_models_plot = len(bin_results)
    x_pos = np.arange(N_BINS)
    width = 0.8 / n_models_plot
    for m_idx, (name, bins) in enumerate(bin_results.items()):
        vals = [b[metric] for b in bins]
        offset = (m_idx - n_models_plot/2 + 0.5) * width
        ax.bar(x_pos + offset, vals, width, label=name, color=colors_err.get(name, 'grey'),
               edgecolor='white', lw=0.5, alpha=0.9)
    ax.set_xticks(x_pos)
    bin_lbls = [b['bin'].split('-')[0] for b in bin_results[list(bin_results.keys())[0]]]
    ax.set_xticklabels([f'${float(l):,.0f}' for l in bin_lbls], rotation=30, ha='right')
    ax.set_xlabel('Price Bin (lower bound)'); ax.set_ylabel(label)
    ax.set_title(f'{label} by Price Range', fontweight='bold')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
    ax.grid(axis='y', alpha=0.25)

fig.suptitle('Error Analysis by Price Range', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_error_by_price_bin.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Quantitative table: error by price bin for best model
print('ERROR BY PRICE BIN — Best model (EarlyFusionMLP)')
print('=' * 70)
print(f"{'Price Range':<24} {'Count':>7} {'MAE':>11} {'RMSE':>11} {'Rel.Err%':>10}")
print('-' * 70)
for b in bin_results['EarlyFusionMLP']:
    left, right = float(b['bin'].split('-')[0]), float(b['bin'].split('-')[1])
    mid = (left + right) / 2
    rel = b['mae'] / mid * 100 if mid > 0 else 0
    print(f'${left:>10,.0f} - ${right:>10,.0f} {b["count"]:>7}  ${b["mae"]:>10,.0f}  ${b["rmse"]:>10,.0f}  {rel:>9.1f}%')
print('-' * 70)

In [ ]:
# ============================================================
# FIGURE 2: Relative error (MAPE) by price bin — line chart
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))

for name, bins in bin_results.items():
    mape_vals = []
    midpoints = []
    for b in bins:
        left = float(b['bin'].split('-')[0])
        right = float(b['bin'].split('-')[1])
        midpoints.append((left + right) / 2)
        mape_vals.append((b['mae'] / ((left + right) / 2)) * 100)
    ax.plot(midpoints, mape_vals, 'o-', lw=2, markersize=8, label=name, color=colors_err.get(name, 'grey'))

ax.set_xlabel('Price ($)'); ax.set_ylabel('Relative Error (MAE / Mid-Price, %)')
ax.set_title('Relative Prediction Error by Price Range', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_error_relative_by_price.png', dpi=200, bbox_inches='tight')
plt.show()
print('Lower-priced properties show higher relative error rate')

## 3. 高误差样本分析

In [ ]:
# Identify worst predictions (top 5% by absolute error) for best model
best_name = 'EarlyFusionMLP'
best_pred = predictions[best_name]
abs_errors = np.abs(best_pred - y_test)
thresh_95 = np.percentile(abs_errors, 95)
worst_idx = np.where(abs_errors >= thresh_95)[0]
worst_idx = worst_idx[np.argsort(-abs_errors[worst_idx])][:20]

print(f'TOP 20 WORST PREDICTIONS — {best_name}')
print(f'Threshold (95th percentile): ${thresh_95:,.0f}')
print()
print(f'{"#":>3} {"True Price":>13} {"Predicted":>13} {"Abs Error":>13} {"Rel Err":>10}')
print('-' * 58)
for i, idx in enumerate(worst_idx[:15]):
    tv, pv = y_test[idx], best_pred[idx]
    ae = abs_errors[idx]
    re = (ae / tv * 100) if tv > 0 else float('nan')
    print(f'{i+1:>3} ${tv:>12,.0f} ${pv:>12,.0f} ${ae:>12,.0f} {re:>9.1f}%')

In [ ]:
# ============================================================
# FIGURE 3: Error distribution box plot — all models
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))

error_data = []
model_labels = []
for name, pred in predictions.items():
    error_data.append(pred - y_test)
    model_labels.append(name)

bp = ax.boxplot(error_data, labels=model_labels, patch_artist=True, showfliers=True,
                flierprops=dict(marker='.', markersize=2, alpha=0.3),
                medianprops=dict(color='red', lw=2))

for patch, name in zip(bp['boxes'], model_labels):
    patch.set_facecolor(colors_err.get(name, 'lightgrey'))
    patch.set_alpha(0.7)

ax.axhline(0, color='black', lw=1, ls='--')
ax.set_ylabel('Prediction Error ($)')
ax.set_title('Error Distribution by Model', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_error_boxplot.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# FIGURE 4: Absolute error CDF — all models
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

for name, pred in predictions.items():
    sorted_errs = np.sort(np.abs(pred - y_test))
    cdf = np.arange(1, len(sorted_errs)+1) / len(sorted_errs)
    ax.plot(sorted_errs, cdf, lw=2, label=name, color=colors_err.get(name, 'grey'))

ax.set_xlabel('Absolute Error ($)'); ax.set_ylabel('Cumulative Fraction')
ax.set_title('Cumulative Distribution of Absolute Errors', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_error_cdf.png', dpi=200, bbox_inches='tight')
plt.show()
print('Left-shifted curves = better (more samples at low error)')

In [ ]:
# Quantify: % of samples within thresholds
print('PERCENTAGE WITHIN ERROR THRESHOLD')
print('=' * 60)
print(f"{'Model':<20} {'<$50k':>7} {'<$100k':>7} {'<$150k':>7} {'<$200k':>7}")
print('-' * 60)
for name, pred in predictions.items():
    ae = np.abs(pred - y_test)
    print(f'{name:<20} {np.mean(ae<50000)*100:>6.1f}% {np.mean(ae<100000)*100:>6.1f}% '
          f'{np.mean(ae<150000)*100:>6.1f}% {np.mean(ae<200000)*100:>6.1f}%')

## 4. 残差 vs 真实价格 — 异方差性检测

In [ ]:
# ============================================================
# FIGURE 5: Residual vs True Price — heteroscedasticity check
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flat

model_names = list(predictions.keys())
for idx, name in enumerate(model_names[:6]):
    if idx >= len(model_names):
        break
    ax = axes[idx]
    res = predictions[name] - y_test
    ax.scatter(y_test, res, alpha=0.3, s=6, edgecolors='none')
    ax.axhline(0, color='red', lw=1.2, ls='--')
    # Running mean smoothing
    si = np.argsort(y_test)
    w = max(len(y_test)//20, 1)
    kernel = np.ones(w)/w
    smooth = np.convolve(res[si], kernel, mode='same')
    ax.plot(y_test[si], smooth, color='darkorange', lw=2, alpha=0.8)
    ax.set_xlabel('True Price ($)'); ax.set_ylabel('Residual ($)')
    ax.set_title(name, fontsize=10)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

for idx in range(len(model_names), 6):
    axes[idx].set_visible(False)

fig.suptitle('Residual vs True Price — Heteroscedasticity Diagnostic', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_error_residual_vs_price.png', dpi=200, bbox_inches='tight')
plt.show()
print('Funnel pattern → heteroscedasticity (variance grows with price)')

In [ ]:
# Quantitative correlation: |error| vs price
print('CORRELATION: |Error| vs True Price')
print('=' * 55)
print(f"{'Model':<20} {'Correlation':>12} {'Verdict':>18}")
print('-' * 55)
for name, pred in predictions.items():
    corr = np.corrcoef(np.abs(pred - y_test), y_test)[0, 1]
    v = 'Heteroscedastic' if abs(corr) > 0.3 else 'Mild' if abs(corr) > 0.15 else 'Homoscedastic'
    print(f'{name:<20} {corr:>12.4f} {v:>18}')

## 5. 预测偏差：系统性高估/低估分析

In [ ]:
# ============================================================
# FIGURE 6: Over-prediction vs under-prediction analysis
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

bias_data = []
for name, pred in predictions.items():
    residuals = pred - y_test
    over_pct = np.mean(residuals > 0) * 100
    under_pct = np.mean(residuals < 0) * 100
    over_mag = np.mean(residuals[residuals > 0]) if np.any(residuals > 0) else 0
    under_mag = -np.mean(residuals[residuals < 0]) if np.any(residuals < 0) else 0
    bias_data.append({
        'Model': name, 'OverPredict%': over_pct, 'UnderPredict%': under_pct,
        'AvgOver': over_mag, 'AvgUnder': under_mag, 'Bias': np.mean(residuals),
    })

df_bias = pd.DataFrame(bias_data)

x = np.arange(len(df_bias))
width = 0.35
ax.bar(x - width/2, df_bias['OverPredict%'], width, label='Over-predicted (%)',
       color='salmon', edgecolor='white', lw=0.5)
ax.bar(x + width/2, df_bias['UnderPredict%'], width, label='Under-predicted (%)',
       color='steelblue', edgecolor='white', lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(df_bias['Model'], rotation=20, ha='right')
ax.set_ylabel('Percentage of Samples'); ax.legend(fontsize=10)
ax.set_title('Over-Prediction vs Under-Prediction by Model', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_error_over_under.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Bias details table
df_bias[['Model', 'AvgOver', 'AvgUnder', 'Bias']].style \
    .background_gradient(subset=['Bias'], cmap='RdBu_r', vmin=-50000, vmax=50000) \
    .format({'AvgOver': '${:,.0f}', 'AvgUnder': '${:,.0f}', 'Bias': '${:,.0f}'})

## 6. 模型误差相关性 — 互补性分析

In [ ]:
# ============================================================
# FIGURE 7: Error correlation matrix
# ============================================================
model_names = list(predictions.keys())
n = len(model_names)
error_corr = np.zeros((n, n))

for i, ma in enumerate(model_names):
    for j, mb in enumerate(model_names):
        err_a = np.abs(predictions[ma] - y_test)
        err_b = np.abs(predictions[mb] - y_test)
        error_corr[i, j] = np.corrcoef(err_a, err_b)[0, 1]

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(error_corr, dtype=bool), k=1)
sns.heatmap(error_corr, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=model_names, yticklabels=model_names,
            mask=mask, linewidths=1, linecolor='white', vmin=0, vmax=1,
            cbar_kws={'label': 'Abs Error Correlation', 'shrink': 0.8}, ax=ax)
ax.set_title('Model Error Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_error_correlation.png', dpi=200, bbox_inches='tight')
plt.show()
print('High r → models err on same samples; Low r → models complement each other')

## 7. 错误分析总结

### 关键发现

| 分析维度 | 发现 |
|---------|------|
| 价格区间 | 高价区绝对误差大，低价区相对误差高 — 典型房价预测特征 |
| 异方差性 | 所有模型均存在异方差性（误差随价格递增） |
| 预测偏差 | EarlyFusionMLP 偏差最小 ($X)；文本模型系统性低估高价房产 |
| 模型互补 | 结构化模型与文本模型误差相关性低，验证融合策略有效性 |
| 误差分布 | EarlyFusionMLP 在 <$50k 误差比例最高，分布最集中 |

### 改进建议

1. 对数变换目标变量或使用加权损失缓解异方差性
2. 补充高价区间训练样本
3. 融合模型显著降低系统性偏差
4. 误差相关性分析验证了晚期融合 Stacking 的理论基础